# Tutorial 3 - Excel Alpha Queue And Payloads

This tutorial covers the Excel schema, settings overrides, metadata columns, payload generation, and duplicate identity hashes.


In [ ]:
from __future__ import annotations

import csv
import json
import os
import shutil
import sqlite3
import subprocess
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Any

RUN_LIVE = os.getenv("BRAIN_SIM_RUN_LIVE") == "1"
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / "examples").exists() else CWD.parent if CWD.name == "examples" else CWD
EXAMPLE_DIR = ROOT / "examples"
DATA_DIR = EXAMPLE_DIR / "data"
EXPECTED_DIR = EXAMPLE_DIR / "expected"
RUNS_DIR = EXAMPLE_DIR / "runs"
RUNS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repository root: {ROOT}")
print(f"Live execution enabled: {RUN_LIVE}")


## 1. Read A Valid Excel Queue

The required column is `expression`. Optional simulation settings override `SimulationSettings`; other columns become metadata.


In [ ]:
from brain_sim.excel import read_excel_expressions
from brain_sim.payloads import build_payload_record

excel_path = DATA_DIR / "tutorial_03_mixed_settings.xlsx"
alpha_rows = read_excel_expressions(excel_path, sheet_name="alphas")
payload_records = []
for index, alpha in enumerate(alpha_rows, start=1):
    record = build_payload_record(alpha)
    payload_records.append(
        record.__class__(
            row_id=record.row_id,
            alpha_hash=f"tutorial-hash-{index}",
            payload=record.payload,
            metadata=record.metadata,
        )
    )

print(f"Loaded {len(payload_records)} payload records from {excel_path.name}")
[(record.row_id, record.alpha_hash, record.payload["settings"]["universe"]) for record in payload_records]


## 2. Inspect Settings And Metadata

`theme` is not a BRAIN setting, so it is preserved as row metadata. Numeric and boolean setting columns are coerced before payload creation.


In [ ]:
first_alpha = alpha_rows[0]
print(first_alpha.settings)
print(first_alpha.metadata)
print(payload_records[0].payload)


## 3. Payload Hash Identity

The duplicate key is a SHA-256 hash of the full normalized simulation payload. Changing the expression or any setting changes the hash.


In [ ]:
from brain_sim.payloads import build_regular_payload, hash_payload, normalize_payload

payload = build_regular_payload(first_alpha)
print(normalize_payload(payload))
print(hash_payload(payload))


## 4. Invalid Excel Schema

Missing `expression` raises `ExcelInputError` before any simulation is submitted.


In [ ]:
from brain_sim.excel import ExcelInputError, read_excel_expressions

invalid_path = DATA_DIR / "tutorial_03_invalid_missing_expression.xlsx"
try:
    read_excel_expressions(invalid_path, sheet_name="alphas")
except ExcelInputError as exc:
    print(type(exc).__name__, str(exc))
